<a href="https://colab.research.google.com/github/diyamofficial/Internship_June2026/blob/main/Phase2_RAG_beforeImprovement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community chromadb sentence-transformers groq ragas datasets langgraph -q
from langchain_community.vectorstores import Chroma

from google.colab import userdata
from groq import Groq

import os

In [ ]:
!pip install wikipedia-api -q

import wikipediaapi
import os

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-project/1.0'
)

os.makedirs("wiki_docs", exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls wiki_docs

In [ ]:
!cp -r wiki_docs /content/drive/MyDrive/

In [ ]:
print(len(os.listdir("wiki_docs")))

In [ ]:
import wikipediaapi
import os

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-project/1.0'
)

os.makedirs("wiki_docs", exist_ok=True)

topics = [
    "Machine learning",
    "Deep learning",
    "Artificial neural network",
    "Convolutional neural network",
    "Recurrent neural network",
    "Long short-term memory",
    "Transformer (machine learning model)",
    "Large language model",
    "Backpropagation",
    "Gradient descent",
    "Computer vision",
    "Image classification",
    "Object detection",
    "Support vector machine",
    "Random forest",
    "Decision tree learning",
    "XGBoost",
    "K-nearest neighbors algorithm"
]

for topic in topics:
    page = wiki.page(topic)

    if page.exists():
        filename = topic.replace(" ", "_").replace("/", "_") + ".txt"

        with open(f"wiki_docs/{filename}", "w", encoding="utf-8") as f:
            f.write(page.text)

        print(f"Saved: {filename}")
    else:
        print(f"Not found: {topic}")

In [ ]:
import os

files = os.listdir("wiki_docs")

print("Number of files:", len(files))
print(files)

In [ ]:
with open("wiki_docs/Machine_learning.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

In [ ]:
!pip show langchain

In [ ]:
!pip install langchain-community -q

In [ ]:
# Install Required Libraries,Import
!pip install langchain chromadb langchain-google-genai -q
!pip install langchain-text-splitters -q
import os

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma


In [ ]:
import os
from langchain_core.documents import Document

documents = []

for filename in os.listdir("wiki_docs"):
    filepath = os.path.join("wiki_docs", filename)

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append(
        Document(
            page_content=text,
            metadata={"source": filename}
        )
    )

print("Documents Loaded:", len(documents))

In [ ]:
print(documents[0].metadata)

print("\nFirst 500 characters:\n")

print(documents[0].page_content[:500])

**Splitting into chunks of 500 words each**

In [ ]:
total_words = sum(len(doc.page_content.split()) for doc in documents)

print("Total words:", total_words)

In [ ]:
from langchain_core.documents import Document

word_chunks = []

for doc in documents:
    words = doc.page_content.split()

    chunk_size = 500
    overlap = 50

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk_text = " ".join(words[start:end])

        word_chunks.append(
            Document(
                page_content=chunk_text,
                metadata=doc.metadata
            )
        )

        start += chunk_size - overlap

print("Total Chunks:", len(word_chunks))

In [ ]:
word_counts = [len(chunk.page_content.split()) for chunk in word_chunks]

print("Total Chunks:", len(word_chunks))
print("Average words:", sum(word_counts)/len(word_counts))
print("Min words:", min(word_counts))
print("Max words:", max(word_counts))

In [ ]:
print(len(word_chunks[0].page_content.split()))

**Embedding Starting**

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
sample_embedding = embeddings.embed_query(
    word_chunks[0].page_content
)

print("Embedding dimension:", len(sample_embedding))

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
test_chunks = word_chunks[:10]

vectorstore = Chroma.from_documents(
    documents=test_chunks,
    embedding=embeddings,
    persist_directory="./test_db"
)

print("Test successful!")

**OPTION 2 - USING a local embedding model**

---



In [ ]:
!pip install sentence-transformers chromadb langchain-community -q
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=word_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Done!")
print("Total indexed:", vectorstore._collection.count())

In [ ]:
# TO LOAD EXISTING CHROMADB
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

**Initially, Gemini embeddings were used for generating vector representations of document chunks. However, during bulk ingestion of large chunks into ChromaDB, repeated API quota and rate-limit issues were encountered. To ensure smooth and scalable indexing, the embedding model was switched to the local Sentence-Transformers model `all-MiniLM-L6-v2`.**

This model generates embeddings locally inside the Colab environment without relying on external API calls, which avoids quota limitations and enables faster large-scale document processing. The generated embeddings are then stored in ChromaDB, where semantic similarity search can be performed efficiently during retrieval in the RAG pipeline.

**Trying with a specific query**

In [ ]:
query = "What is backpropagation?"

results = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}\n")
    print(doc.page_content[:500])

**Adding Dynamic user input**

In [ ]:
!pip install groq
from google.colab import userdata
from groq import Groq
import os

try:
    # Load GROQ API KEY
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_KEY")
    # Create Groq Client
    client = Groq(
        api_key=os.environ["GROQ_API_KEY"]
    )

    # Take User Query
    query = input("Enter your question: ")

    # Empty query check
    if not query.strip():
        raise ValueError("Query cannot be empty.")

    # Retrieve Top-5 Similar Chunks
    results = vectorstore.similarity_search_with_score(
        query,
        k=5
    )

    # Filter Relevant Chunks
    relevant_docs = []

    print("\nTop Retrieved Chunks:\n")

    for i, (doc, score) in enumerate(results, start=1):

        # Lower score = better similarity
        if score < 1.0:

            relevant_docs.append(doc)

            print(f"\nResult {i}")
            print(f"Similarity Score: {score}\n")

            print(doc.page_content[:500])

    # Check if Relevant Docs Exist
    if not relevant_docs:
        raise ValueError("No sufficiently relevant documents found.")

    # Create Context
    context = "\n\n".join(
        [doc.page_content for doc in relevant_docs]
    )

    # Create Prompt
    prompt = f"""
    Answer the question using ONLY the context below.

    If the answer is not present in the context,
    clearly say that the information is unavailable.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # Generate Response from Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    # Print Final Answer
    print("\nFinal Answer:\n")
    print(response.choices[0].message.content)

# Exception Handling
except Exception as e:

    print("\nError occurred:")
    print(e)

**LangGraph Workflow Pipeline**

In [ ]:
!pip install langgraph -q
from typing import TypedDict
from langgraph.graph import StateGraph, END

In [ ]:
# Define Shared state
class RAGState(TypedDict):

    query: str
    retrieved_docs: list
    context: str
    answer: str

In [ ]:
# Retrieval Node
def retrieve(state):

    try:

        query = state["query"]

        results = vectorstore.similarity_search_with_score(
            query,
            k=5
        )

        relevant_docs = []

        for doc, score in results:

            if score < 1.0:
                relevant_docs.append(doc)

        if not relevant_docs:
            raise ValueError("No relevant documents found.")

        return {
            "retrieved_docs": relevant_docs
        }

    except Exception as e:

        print("Retrieval Error:", e)

        return {
            "retrieved_docs": []
        }

In [ ]:
# Context Builder Node

def build_context(state):

    try:

        docs = state["retrieved_docs"]

        context = "\n\n".join(
            [doc.page_content for doc in docs]
        )

        return {
            "context": context
        }

    except Exception as e:

        print("Context Error:", e)

        return {
            "context": ""
        }

In [ ]:
# Generation Node
def generate_answer(state):

    try:

        query = state["query"]
        context = state["context"]

        prompt = f"""
        Answer the question using ONLY the context below.

        If the answer is unavailable in the context,
        clearly mention that.

        Context:
        {context}

        Question:
        {query}

        Answer:
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        answer = response.choices[0].message.content

        return {
            "answer": answer
        }

    except Exception as e:

        print("Generation Error:", e)

        return {
            "answer": "Error generating answer."
        }

In [ ]:
# Create Graph
graph = StateGraph(RAGState)

# Add Nodes
graph.add_node("retrieve", retrieve)

graph.add_node("build_context", build_context)

graph.add_node("generate_answer", generate_answer)

# Define Flow
graph.set_entry_point("retrieve")

graph.add_edge("retrieve", "build_context")

graph.add_edge("build_context", "generate_answer")

graph.add_edge("generate_answer", END)

# Compile Graph
rag_graph = graph.compile()

print("LangGraph pipeline created successfully!")

In [ ]:
query = input("Enter your question: ")

result = rag_graph.invoke(
    {
        "query": query
    }
)

print("\nFinal Answer:\n")

print(result["answer"])

**RAGAS EVALUATION**

In [ ]:
!pip install ragas datasets -q
from datasets import Dataset

In [ ]:
questions = [

    "Why are CNNs considered more effective for image processing tasks compared to traditional neural networks?",

    "How does backpropagation help a neural network improve its predictions over time?",

    "Why is gradient descent important during the training of machine learning models?",

    "What problem does LSTM solve that traditional recurrent neural networks struggle with?",

    "How do convolutional layers help CNNs identify patterns in images?",

    "Why is the vanishing gradient problem a major issue in deep neural networks?",

    "How are transformers different from CNNs in handling data and learning patterns?",

    "Why do neural networks require activation functions instead of simple linear operations?",

    "How does semantic similarity search improve information retrieval in RAG systems?",

    "Why can overfitting reduce the real-world performance of a machine learning model?",

    "How does chunking affect the performance and accuracy of a RAG pipeline?",

    "Why are embeddings important in vector databases such as ChromaDB?",

    "How does a CNN learn low-level and high-level image features during training?",

    "Why is context retrieval necessary before generating answers in a RAG system?",

    "How do attention mechanisms improve the performance of large language models?",

    "Why are pretrained models commonly used in modern deep learning applications?",

    "How does similarity scoring help filter irrelevant chunks during retrieval?",

    "Why is exception handling important in AI pipelines and workflow systems?",

    "How does LangGraph help organize and manage a multi-step RAG workflow?",

    "Why can a RAG system sometimes produce incorrect or incomplete answers even after retrieval?"

]

In [ ]:
answers = []
contexts = []

for question in questions:

    try:

        result = rag_graph.invoke(
            {
                "query": question
            }
        )

        answers.append(result["answer"])

        retrieved = vectorstore.similarity_search(
            question,
            k=3
        )

        contexts.append(
            [doc.page_content for doc in retrieved]
        )

        print(f"Completed: {question}")

    except Exception as e:

        print(f"Error for question: {question}")

        print(e)

        answers.append("Error")

        contexts.append([])

In [ ]:
ground_truths = [

    "CNNs are effective for image processing because they capture spatial features using convolutional layers.",

    "Backpropagation updates neural network weights by computing gradients.",

    "Gradient descent minimizes loss by iteratively updating parameters.",

    "LSTM solves long-term dependency problems in sequence learning.",

    "Convolutional layers detect important image patterns and features.",

    "Vanishing gradients make deep neural network training difficult.",

    "Transformers use attention mechanisms while CNNs use convolutions.",

    "Activation functions help neural networks learn non-linear relationships.",

    "Semantic similarity search retrieves contextually related information.",

    "Overfitting causes poor generalization on unseen data.",

    "Chunking affects retrieval quality and context relevance in RAG systems.",

    "Embeddings convert text into vector representations for similarity search.",

    "CNNs progressively learn simple and complex image features.",

    "Context retrieval grounds answers using external information.",

    "Attention mechanisms help models focus on important information.",

    "Pretrained models reduce training time and improve performance.",

    "Similarity scoring filters less relevant retrieval results.",

    "Exception handling prevents workflow failures and improves robustness.",

    "LangGraph organizes AI workflows using graph-based execution.",

    "RAG systems may fail because of weak retrieval or incomplete context."

]

In [ ]:
dataset = Dataset.from_dict(
    {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    }
)

In [ ]:
evaluation_results = []

for i in range(len(questions)):

    question = questions[i]
    answer = answers[i]
    context = contexts[i]

    evaluation_prompt = f"""
    You are evaluating a RAG system.

    Evaluate the following answer based on:

    1. Faithfulness:
    Is the answer supported by the retrieved context?

    2. Answer Relevancy:
    Does the answer properly address the question?

    3. Context Quality:
    Was the retrieved context useful?

    Give scores out of 10.

    Also provide a short explanation.

    Question:
    {question}

    Retrieved Context:
    {context}

    Generated Answer:
    {answer}
    """

    try:

        response = client.chat.completions.create(

            model="llama-3.3-70b-versatile",

            messages=[
                {
                    "role": "user",
                    "content": evaluation_prompt
                }
            ]
        )

        evaluation = response.choices[0].message.content

        evaluation_results.append({

            "question": question,

            "evaluation": evaluation
        })

        print(f"\nCompleted Evaluation: {i+1}")

    except Exception as e:

        print(f"\nEvaluation Error at question {i+1}")
        print(e)

In [ ]:
for item in evaluation_results:

    print("\nQUESTION:\n")
    print(item["question"])

    print("\nEVALUATION:\n")
    print(item["evaluation"])

    print("\n" + "="*80)